# Labels and buckets with CASE

Practice notebook for the datafromzero SQL lesson **Labels and buckets with CASE**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Look at the prices

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd
FROM products
ORDER BY unit_price_usd DESC, product_name
""").show()

### Step 2. A first CASE with two outcomes

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd,
       CASE WHEN unit_price_usd >= 4.50 THEN 'premium'
            ELSE 'regular'
       END AS price_tier
FROM products
""").show()

### Step 3. Add the middle tier

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd,
       CASE WHEN unit_price_usd >= 4.50 THEN 'premium'
            WHEN unit_price_usd >= 3.50 THEN 'standard'
            ELSE 'budget'
       END AS price_tier
FROM products
ORDER BY unit_price_usd DESC, product_name
""").show()

### Step 4. Predict what no match returns

Before you run it: how many rows show a tier, and what do the others show?

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd,
       CASE WHEN unit_price_usd >= 4.50 THEN 'premium' END AS price_tier
FROM products
ORDER BY unit_price_usd DESC, product_name
""").show()

### Step 5. Two conditions, as in E6

In [ ]:
con.sql("""
SELECT order_id, payment_method, status,
       CASE WHEN payment_method = 'cash' AND status = 'completed' THEN 'in till'
            ELSE 'skip'
       END AS till_flag
FROM orders
WHERE order_id IN (1458, 1461)
""").show()

### Step 6. Verify with a second route

Count the same groups with `WHERE`, not `CASE`.

In [ ]:
con.sql("""
SELECT COUNT(*) AS premium_products FROM products WHERE unit_price_usd >= 4.50
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS in_till FROM orders WHERE payment_method = 'cash' AND status = 'completed'
""").show()

## Exercise 1. Drink or food

The first lines are given. Add the branch for everything else, close the `CASE` with the name `menu_section`, and
name the table.

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    SELECT product_name, category,
           CASE WHEN category IN ('coffee', 'tea') THEN 'drink'
    """).show()
except Exception as e:
    print(e)


## Exercise 2. Size the order lines and show the three biggest

Before you run anything, write one sentence in the first cell: which label do you expect on the first row? Then label
each line `single` (1 unit), `few` (2 or 3) or `bulk` (4 or more), and show the three lines with the largest
`quantity`, with `order_item_id` as the tie-breaker.

In [ ]:
# My expected result, in words:
#

In [ ]:
# My query:

## Exercise 3. Spot the flaw

A colleague writes the price tiers with the tests in this order. The query runs without an error. Run it and compare
the tiers with the price list: which tier is missing, and why? Then write a corrected query.

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd,
       CASE WHEN unit_price_usd >= 3.50 THEN 'standard'
            WHEN unit_price_usd >= 4.50 THEN 'premium'
            ELSE 'budget'
       END AS price_tier
FROM products
ORDER BY unit_price_usd DESC, product_name
""").show()

In [ ]:
# Your corrected query: